**VGG by Oxford**- depth is critical for good performance , known for simplicity and homogenity
**Resnet by MS research** - rev. for skip connections, adresses vanishing gradient, of sizes 50,101,152 layers

VGG trains faster but resnet does better


In [5]:
# simple starightfwd task - vgg
import torch
import torch.nn as nn

class VGG(nn.Module):
  def __init__(self,num_classes=1000):
    super().__init__()
    self.features = nn.Sequential(
        # 1st block
        nn.Conv2d(3,64,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64,64,kernel_size=3,padding=1),
        nn.ReLU(inplace=True)
        nn.MaxPool2d(kernel_size=2,stride=2),
        # 2nd block
        nn.Conv2d(64,128,kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # 3rd block
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # 4th block
        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # 5th block
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        )
    self.classifier = nn.Sequential(
        nn.Linear(512*7*7,4096),
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(4096,4096),
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(4096,num_classes),
    )
   def forward(self,x):
      x = self.features(x)
      x = x.view(x.size(0),-1)
      x = self.classifier(x)
      return x


In [9]:
# resnet in pytorch
import torch
import torch.nn as nn
# self.in_channels is a Dynamic State
class ResidualBlock(nn.Module):
  def __init__(self,in_channels,out_channels,stride=1,downsample=None):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.downsample = downsample # matches dimensions to add to original input
  def forward(self,x):
    residual = x
    if self.downsample is not None:
      residual = self.downsample(x)
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    out += residual
    out = self.relu(out)
    return out
class ResNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.in_channels = 64
    self.conv1 = nn.Conv2d(3,64,kernel_size=7, stride = 2, padding=3)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3,stride=2,padding=3)
    self.layer1 = self.make_layer(64,2)
    self.layer2 = self.make_layer(128,2,2)
    self.layer3 = self.make_layer(256,2,2)
    self.layer4 = self.make_layer(512,2,2)
    self.avgpool = nn.AdaptiveAvgPool2d((1,1)) # global avg pooling
    self.fc = nn.Linear(512,10)
  def make_layer(self, out_channels,blocks,stride=1):
    downsample = None
    if stride !=1 or self.in_channels != out_channels:
      downsample = nn.Sequential(
          nn.Conv2d(self.in_channels,out_channels,kernel_size=1,stride=stride),
          nn.BatchNorm2d(out_channels),
      )
    layers =[]
    layers.append(ResidualBlock(self.in_channels,out_channels,stride,downsample))
    return nn.Sequential(*layers)

def forward(self,x):
  out = self.conv1(x)
  out= self.bn1(out)
  out = self.relu(out)
  out = self.maxpool(out)
  out = self.layer1(out)
  out = self.layer2(out)
  out = self.layer3(out)
  out = self.layer4(out)
  out = self.avgpool(out)
  out = out.view(out.size(0),-1)
  out = self.fc(out)
  return out


residual block is the building block of resnet as u can see above, self.in_channals is dynamic, when make_layer runs it checks if it needs to downsampled or not and then makes one if needed, and creates  a residual block.

In [ ]:
# regularization and optimization in CNNs
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation
transforms = Compose([
RandomHorizontalFlip(),
RandomRotation(10),
])

# add regularisation to penalize large weights
import torch.nn as nn
model = nn.Sequential(
nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3),
nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
nn.Flatten(),
nn.Linear(32 * 14 * 14, 10),
)
criterion = nn.CrossEntropyLoss() + 0.01 * torch.norm(model.parameters(),p=2)

In [ ]:
# grid search - technique for hyperparameter tuning
from sklearn.model-selection import GridSearchCV
from skorch import NeuralNetClassifier
params = {
    'lr': [0.01,0.001],
    'max_epochs' :[10,20]
}
net = NeuralNetClassifier(model,criterion=criterion,optmizer=torch.optim.Adam)
gs = GridSearchCV(net,params,refit=False,cv=3,scoring = 'accuracy')
gs.fit(X_train, y_train)

In [ ]:
from torchvision import models
#load pretrained models
model = models.resnet18(pretrained=True)
# fine -tune the last layer
for param in model.parameters():
  param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features,num_classes)

In [ ]:
# learning rate schedule
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer,step_size=30, gamma = 0.1)
# use batch norm
nn.BatchNorm2d(num_features) # to avoid cascading of weight value ranges btw layers
# to avoid internal covariate shift this was introduced

# do proper weight initialisation
nn.init.xavier_uniform(layer.weight)
# cross val
from sklearn.model_selection import cross_val_score
cross_val_score(net,X_train, y_train, cv=3)